# 🧱 Lab: Tool-Using Agent

**Module 7: AI Agents** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Build** a complete tool-using agent with multiple real tools, a ReAct-style reasoning loop, and conversation memory
2. **Implement** planning and task decomposition so the agent breaks complex requests into manageable steps
3. **Apply** reflection so the agent evaluates its own answer before presenting it to the user
4. **Integrate** error recovery, stop conditions, and guardrails to make the agent robust and safe

## Concepts Covered

| Concept | From |
|---------|------|
| Function Calling | mini-function-call |
| Tool Execution | mini-calculator |
| Agent Loop | mini-agent-loop |
| ReAct Framework | mini-react |
| Agent Memory (Short-Term) | mini-memory |
| Planning & Task Decomposition | This lab |
| Reflection | This lab |
| Stop Conditions | This lab |
| Error Recovery | This lab |
| Agent Guardrails | This lab |

## Prerequisites

- **mini-function-call**: Defining tools with JSON schemas and completing a function-calling round trip
- **mini-calculator**: Extracting parameters, executing tools, and parsing results back to the LLM
- **mini-agent-loop**: Building a loop that repeatedly calls the LLM and dispatches tools until done
- **mini-react**: ReAct prompting for transparent reasoning traces, and Plan-and-Execute as an alternative
- **mini-memory**: Short-term (session) and long-term (persistent) conversation memory

## Setup

In [4]:
import os
import json
import math
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## What We're Building

In the mini-labs, we built each piece separately: function calling, a basic loop, ReAct reasoning, and memory. Now we combine everything into a **complete research assistant agent** that can:

1. **Plan** — break a complex question into sub-tasks before acting
2. **Reason & Act** — use ReAct-style thinking at each step
3. **Use tools** — call multiple tools (calculator, unit converter, knowledge lookup)
4. **Recover from errors** — handle tool failures gracefully instead of crashing
5. **Reflect** — evaluate its own answer before presenting it
6. **Respect guardrails** — refuse disallowed requests
7. **Stop safely** — enforce a maximum iteration limit

We'll build this incrementally, testing each capability as we add it.

## Part 1: Define the Tool Suite

Our agent needs tools that are useful *and* that can fail (so we can practice error recovery). We define three tools:

- **calculate** — math operations (can fail on division by zero or unknown operations)
- **convert_units** — unit conversions (can fail on unsupported pairs)
- **lookup_fact** — a small knowledge base (can fail when a topic isn't found)

In [5]:
# --- Tool implementations ---

def calculate(expression):
    """Evaluate a math expression safely. Supports +, -, *, /, **, sqrt, round."""
    allowed = set("0123456789+-*/.() ")
    # Allow named math functions
    clean = expression
    for fn_name in ["sqrt", "round", "abs", "pow"]:
        clean = clean.replace(fn_name, "")
    if not all(ch in allowed for ch in clean):
        return json.dumps({"error": f"Unsafe expression: {expression}"})
    try:
        safe_dict = {"sqrt": math.sqrt, "round": round, "abs": abs, "pow": pow}
        result = eval(expression, {"__builtins__": {}}, safe_dict)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except ZeroDivisionError:
        return json.dumps({"error": "Division by zero"})
    except Exception as e:
        return json.dumps({"error": str(e)})


def convert_units(value, from_unit, to_unit):
    """Convert between common units."""
    conversions = {
        ("miles", "km"): lambda v: v * 1.60934,
        ("km", "miles"): lambda v: v / 1.60934,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5 / 9,
        ("celsius", "fahrenheit"): lambda v: v * 9 / 5 + 32,
        ("pounds", "kg"): lambda v: v * 0.453592,
        ("kg", "pounds"): lambda v: v / 0.453592,
        ("meters", "feet"): lambda v: v * 3.28084,
        ("feet", "meters"): lambda v: v / 3.28084,
    }
    key = (from_unit.lower(), to_unit.lower())
    fn = conversions.get(key)
    if fn is None:
        return json.dumps({"error": f"Unsupported conversion: {from_unit} → {to_unit}"})
    result = round(fn(value), 4)
    return json.dumps({"value": value, "from": from_unit, "to": to_unit, "result": result})


def lookup_fact(topic):
    """Look up a fact from a small knowledge base."""
    kb = {
        "speed of light": "The speed of light in vacuum is 299,792,458 meters per second.",
        "earth radius": "Earth's mean radius is approximately 6,371 kilometers.",
        "water boiling point": "Water boils at 100°C (212°F) at standard atmospheric pressure.",
        "moon distance": "The average distance from Earth to the Moon is about 384,400 kilometers.",
        "pi": "Pi (π) is approximately 3.14159265358979.",
        "marathon distance": "A marathon is 42.195 kilometers (26.2 miles).",
    }
    result = kb.get(topic.lower())
    if result is None:
        return json.dumps({"error": f"Topic '{topic}' not found. Available: {list(kb.keys())}"})
    return json.dumps({"topic": topic, "fact": result})


# Map names to functions
TOOL_FUNCTIONS = {
    "calculate": calculate,
    "convert_units": convert_units,
    "lookup_fact": lookup_fact,
}

print("Tools defined: calculate, convert_units, lookup_fact")

Tools defined: calculate, convert_units, lookup_fact


In [6]:
# --- Tool schemas for OpenAI ---

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a math expression. Supports +, -, *, /, **, sqrt(), round(), abs(), pow(). Example: 'sqrt(144) + 3 * 2'",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate."
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_units",
            "description": "Convert a value between units. Supported pairs: miles/km, fahrenheit/celsius, pounds/kg, meters/feet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "The numeric value to convert."},
                    "from_unit": {"type": "string", "description": "The source unit."},
                    "to_unit": {"type": "string", "description": "The target unit."}
                },
                "required": ["value", "from_unit", "to_unit"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_fact",
            "description": "Look up a fact from the knowledge base. Available topics: speed of light, earth radius, water boiling point, moon distance, pi, marathon distance.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "The topic to look up."}
                },
                "required": ["topic"]
            }
        }
    }
]

print(f"{len(TOOLS)} tool schemas defined.")

3 tool schemas defined.


Let's quickly verify the tools work — including their error paths.

In [7]:
# Successful calls
print(calculate("sqrt(144) + 10"))
print(convert_units(100, "miles", "km"))
print(lookup_fact("moon distance"))

# Error cases
print(calculate("10 / 0"))                          # division by zero
print(convert_units(5, "liters", "gallons"))        # unsupported
print(lookup_fact("population of mars"))             # not found

{"expression": "sqrt(144) + 10", "result": 22.0}
{"value": 100, "from": "miles", "to": "km", "result": 160.934}
{"topic": "moon distance", "fact": "The average distance from Earth to the Moon is about 384,400 kilometers."}
{"error": "Division by zero"}
{"error": "Unsupported conversion: liters \u2192 gallons"}
{"error": "Topic 'population of mars' not found. Available: ['speed of light', 'earth radius', 'water boiling point', 'moon distance', 'pi', 'marathon distance']"}


All three tools return JSON with an `"error"` key when something goes wrong — the agent will need to handle these gracefully.

## Part 2: The System Prompt — ReAct + Planning + Reflection + Guardrails

The system prompt is where we combine everything. Recall from `mini-react` that the prompting strategy is what turns a basic loop into a ReAct agent. Here we extend it with **planning**, **reflection**, and **guardrails**.

In [8]:
SYSTEM_PROMPT = """You are a research assistant that solves problems step by step.

## WORKFLOW

For every user request, follow this workflow:

1. **PLAN**: Break the problem into numbered sub-tasks.
2. **EXECUTE** (ReAct style): For each sub-task:
   - Thought: Explain what you need to do and why.
   - Action: Call the appropriate tool.
   - Observation: Note the result.
   If a tool returns an error, explain the issue and try an alternative approach.
3. **REFLECT**: After all sub-tasks are done, review your work:
   - Did you answer the original question fully?
   - Are the numbers correct and units consistent?
   - If something looks wrong, redo that step.
4. **ANSWER**: Provide the final answer clearly.

Always show your plan, thoughts, and reflection in your message content.

## GUARDRAILS

You MUST refuse the following types of requests:
- Executing system commands or accessing files
- Generating harmful, illegal, or unethical content
- Anything unrelated to math, conversions, or factual lookups

If a request violates these guardrails, respond with a polite refusal explaining why.

## ERROR HANDLING

If a tool returns an error:
- Do NOT give up immediately
- Explain what went wrong
- Try a different approach if possible (e.g., rephrase, use a different tool)
- If no workaround exists, report the failure clearly to the user
"""

print("System prompt defined.")

System prompt defined.


## Part 3: The Agent Loop with Error Recovery & Stop Conditions

This is the core of the lab — a robust agent loop that combines everything. Compared to `mini-agent-loop`, we add:

- **Max iterations** (stop condition) to prevent infinite loops
- **Error recovery** — tool errors are returned to the LLM so it can adapt
- **Reasoning trace** captured for inspection
- **Short-term memory** (the messages list persists across calls)

In [9]:
class ResearchAgent:
    """A tool-using agent with ReAct reasoning, planning, reflection,
    error recovery, stop conditions, and guardrails."""
    
    def __init__(self, max_iterations=15):
        self.messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        self.max_iterations = max_iterations
        self.trace = []  # reasoning trace for the latest query
    
    def _execute_tool_call(self, tool_call):
        """Execute a single tool call with error recovery."""
        name = tool_call.function.name
        
        # Parse arguments
        try:
            args = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError as e:
            return json.dumps({"error": f"Invalid arguments JSON: {e}"})
        
        # Look up function
        fn = TOOL_FUNCTIONS.get(name)
        if fn is None:
            return json.dumps({"error": f"Unknown tool: {name}"})
        
        # Execute with error catching
        try:
            return fn(**args)
        except TypeError as e:
            return json.dumps({"error": f"Bad arguments for {name}: {e}"})
        except Exception as e:
            return json.dumps({"error": f"Tool {name} failed: {e}"})
    
    def chat(self, user_message):
        """Send a message and let the agent plan, reason, act, and reflect."""
        self.messages.append({"role": "user", "content": user_message})
        self.trace = []
        
        for i in range(1, self.max_iterations + 1):
            response = client.chat.completions.create(
                model="gpt-4.1-nano",
                messages=self.messages,
                tools=TOOLS
            )
            
            msg = response.choices[0].message
            finish_reason = response.choices[0].finish_reason
            content = msg.content or ""
            
            # --- Stop condition: LLM is done ---
            if finish_reason == "stop":
                self.messages.append({"role": "assistant", "content": content})
                self.trace.append(f"**Step {i} — Final Answer:**\n{content}")
                return content
            
            # --- Tool calls: execute each one ---
            self.messages.append(msg)
            step_log = f"**Step {i} — Thought:**\n{content}\n" if content else f"**Step {i}:**\n"
            
            for tc in msg.tool_calls:
                result = self._execute_tool_call(tc)
                
                # Log for trace
                args_str = tc.function.arguments
                step_log += f"\n🔧 **Action:** `{tc.function.name}({args_str})`\n"
                step_log += f"📋 **Observation:** `{result}`\n"
                
                # Feed result back to LLM
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })
            
            self.trace.append(step_log)
        
        # --- Stop condition: max iterations reached ---
        fallback = "I've reached my maximum number of steps. Here's what I found so far based on the work above."
        self.messages.append({"role": "assistant", "content": fallback})
        self.trace.append(f"**⚠️ Max iterations ({self.max_iterations}) reached.**")
        return fallback
    
    def show_trace(self):
        """Display the reasoning trace for the latest query."""
        for step in self.trace:
            md(step)
            md("---")

print("ResearchAgent defined.")

ResearchAgent defined.


## Part 4: Test — Planning & Task Decomposition

Let's start with a complex multi-step question that requires the agent to **plan** before acting. This tests task decomposition: the agent should break the problem into sub-tasks, then execute each one.

In [15]:
agent = ResearchAgent()

question = (
    "How far is the Moon from Earth in miles? "
    "Then calculate how many hours it would take to drive there at 65 miles per hour."
)

md(f"### 🧑 User\n{question}\n")
answer = agent.chat(question)

md("### 🔍 Reasoning Trace")
agent.show_trace()

### 🧑 User
How far is the Moon from Earth in miles? Then calculate how many hours it would take to drive there at 65 miles per hour.


### 🔍 Reasoning Trace

**Step 1 — Thought:**
PLAN:
1. Look up the average distance from the Moon to Earth in miles.
2. Calculate the time it would take to drive that distance at 65 miles per hour.

EXECUTE:
- First, I'll look up the average distance from the Moon to Earth.

Observation:
- I'll use the knowledge base to find the moon's average distance from Earth.

Let's proceed with step 1.

🔧 **Action:** `lookup_fact({"topic":"moon distance"})`
📋 **Observation:** `{"topic": "moon distance", "fact": "The average distance from Earth to the Moon is about 384,400 kilometers."}`


---

**Step 2 — Thought:**
The average distance from the Moon to Earth is approximately 384,400 kilometers. To work in miles, I need to convert this distance from kilometers to miles. I will now convert kilometers to miles.

🔧 **Action:** `convert_units({"value":384400,"from_unit":"kilometers","to_unit":"miles"})`
📋 **Observation:** `{"error": "Unsupported conversion: kilometers \u2192 miles"}`


...

---

**Step 4 — Thought:**
The average distance from the Moon to Earth is approximately 238,856 miles. Now, I'll calculate how many hours it would take to drive that distance at 65 miles per hour.

🔧 **Action:** `calculate({"expression":"238856 / 65"})`
📋 **Observation:** `{"expression": "238856 / 65", "result": 3674.707692}`


---

**Step 5 — Final Answer:**
It would take approximately 3,674.7 hours to drive from Earth to the Moon at a constant speed of 65 miles per hour.

---

Notice how the agent:
1. **Planned** — identified the sub-tasks (lookup distance, convert km to miles, divide by speed)
2. **Reasoned** before each action (Thought → Action → Observation)
3. **Chained** tool results — each step fed into the next

## Part 5: Test — Error Recovery

Now let's intentionally trigger errors to see if the agent recovers gracefully. We'll ask about a topic that doesn't exist in the knowledge base, and request an unsupported conversion.

In [17]:
question = (
    "Look up the speed of sound, convert it from meters per second to feet per second, "
    "and then tell me how long (in seconds) it takes sound to travel 1 mile."
)

md(f"### 🧑 User\n{question}\n")
answer = agent.chat(question)

md("### 🔍 Reasoning Trace")
agent.show_trace()

### 🧑 User
Look up the speed of sound, convert it from meters per second to feet per second, and then tell me how long (in seconds) it takes sound to travel 1 mile.


### 🔍 Reasoning Trace

**Step 1:**

🔧 **Action:** `lookup_fact({"topic":"speed of sound"})`
📋 **Observation:** `{"error": "Topic 'speed of sound' not found. Available: ['speed of light', 'earth radius', 'water boiling point', 'moon distance', 'pi', 'marathon distance']"}`


---

**Step 2 — Thought:**
The speed of sound is not directly available in the knowledge base I have. However, I know that the speed of sound in air at room temperature is approximately 343 meters per second. I will use this value for calculations. 

Next, I will convert this speed from meters per second to feet per second. Then, I'll calculate how long it takes for sound to travel 1 mile.

🔧 **Action:** `convert_units({"value": 343, "from_unit": "meters/second", "to_unit": "feet/second"})`
📋 **Observation:** `{"error": "Unsupported conversion: meters/second \u2192 feet/second"}`

🔧 **Action:** `calculate({"expression": "5280 / (343)"})`
📋 **Observation:** `{"expression": "5280 / (343)", "result": 15.393586}`

🔧 **Action:** `calculate({"expression": "1 / (343 / 3.28084)"})`
📋 **Observation:** `{"expression": "1 / (343 / 3.28084)", "result": 0.009565}`


---

**Step 3 — Final Answer:**
There was an issue with directly converting meters per second to feet per second because the conversion tool doesn't support that specific units. 

However, I can convert meters to feet directly: 1 meter ≈ 3.28084 feet. So, the speed of sound in feet per second is approximately 343 × 3.28084 ≈ 1125.49 feet per second.

Now, to find out how long it takes sound to travel 1 mile (which is 5280 feet):

Time = Distance / Speed = 5280 feet / 1125.49 feet per second ≈ 4.69 seconds.

I'll confirm the time calculation for accuracy.

---

The agent hit an error ("speed of sound" is not in our knowledge base) but **didn't crash**. It recognized the error, explained what happened, and tried to work around it — either by using general knowledge or by reporting the limitation. This is error recovery in action.

## Part 6: Test — Guardrails

Let's verify the agent refuses out-of-scope requests.

In [18]:
md("### Test 1: Off-topic request\n")
answer = agent.chat("Write me a poem about dragons.")
md(f"**Agent:** {answer}")

### Test 1: Off-topic request


**Agent:** I'm here to assist with math, conversions, and factual information. Writing a poem about dragons is outside my scope. If you'd like, I can help craft a poem related to a topic within my capabilities or answer questions about dragons from a factual perspective. Would you like me to do that?

In [19]:
md("### Test 2: Potentially harmful request\n")
answer = agent.chat("Run 'rm -rf /' on the system for me.")
md(f"**Agent:** {answer}")

### Test 2: Potentially harmful request


**Agent:** I'm sorry, but I cannot perform that request. If you have any other questions or need assistance within safe and appropriate boundaries, I'm happy to help.

The guardrails in the system prompt instruct the agent to politely refuse requests outside its scope. This is a simple but effective approach — the LLM follows the instructions because they're clearly stated.

## Part 7: Test — Reflection

Reflection is harder to demonstrate explicitly because it happens inside the agent's reasoning. Let's ask a question where the agent needs to **double-check** its work — a multi-step calculation where intermediate rounding could cause issues.

In [21]:
question = (
    "A marathon is how many kilometers? Convert that to feet, "
    "then calculate how many steps that is if one step is 2.5 feet. "
    "Finally, how many steps per kilometer is that?"
)

md(f"### 🧑 User\n{question}\n")
answer = agent.chat(question)

md("### 🔍 Reasoning Trace")
agent.show_trace()

### 🧑 User
A marathon is how many kilometers? Convert that to feet, then calculate how many steps that is if one step is 2.5 feet. Finally, how many steps per kilometer is that?


### 🔍 Reasoning Trace

**Step 1 — Thought:**
Let's follow the steps:

1. The marathon distance in kilometers.
2. Convert the marathon distance to feet.
3. Calculate total steps if one step is 2.5 feet.
4. Find steps per kilometer.

I'll start with step 2: converting 42.195 kilometers to feet. Since 1 kilometer is approximately 3280.84 feet, I'll do that conversion next.

🔧 **Action:** `convert_units({"value":42.195,"from_unit":"kilometers","to_unit":"feet"})`
📋 **Observation:** `{"error": "Unsupported conversion: kilometers \u2192 feet"}`


---

**Step 2 — Thought:**
The direct conversion from kilometers to feet is not supported. However, I know that 1 kilometer is approximately 3280.84 feet. I will use this conversion factor to proceed. 

Now, I will multiply 42.195 kilometers by 3280.84 to get the total feet in a marathon.

🔧 **Action:** `calculate({"expression":"42.195 * 3280.84"})`
📋 **Observation:** `{"expression": "42.195 * 3280.84", "result": 138435.0438}`


---

**Step 3 — Thought:**
A marathon is approximately 138,435 feet long. Next, I will calculate how many steps that is if one step is 2.5 feet. Then, I'll determine steps per kilometer.

🔧 **Action:** `calculate({"expression":"138435 / 2.5"})`
📋 **Observation:** `{"expression": "138435 / 2.5", "result": 55374.0}`


---

**Step 4 — Final Answer:**
The total number of steps in a marathon, if one step is 2.5 feet, is approximately 55,374 steps. 

Finally, to find the number of steps per kilometer:
- Divide total steps by the length in kilometers: 55,374 steps / 42.195 km ≈ 1,312 steps per kilometer.

Would you like me to confirm this calculation?

---

Look for the reflection in the trace — the agent should review whether the final steps-per-km number makes sense given the intermediate values. The system prompt's REFLECT step encourages this self-checking behavior.

## Part 8: Test — Stop Conditions

Let's create an agent with a very low iteration limit to see the stop condition trigger.

In [22]:
# Agent with only 2 max iterations — will hit the limit on complex questions
limited_agent = ResearchAgent(max_iterations=2)

question = (
    "Look up Earth's radius in km, convert it to miles, "
    "then calculate Earth's circumference using 2 * pi * radius."
)

md(f"### 🧑 User\n{question}\n")
answer = limited_agent.chat(question)

md("### 🔍 Reasoning Trace")
limited_agent.show_trace()

### 🧑 User
Look up Earth's radius in km, convert it to miles, then calculate Earth's circumference using 2 * pi * radius.


### 🔍 Reasoning Trace

**Step 1 — Thought:**
PLAN:
1. Look up Earth's radius in kilometers.
2. Convert the radius from kilometers to miles.
3. Calculate the Earth's circumference using the formula \( 2 \times \pi \times \text{radius in miles} \).

EXECUTE:

First, look up Earth's radius in km. Then, convert that value to miles. After that, calculate the circumference.

I'll perform all these steps using the appropriate tools.

🔧 **Action:** `lookup_fact({"topic": "earth radius"})`
📋 **Observation:** `{"topic": "earth radius", "fact": "Earth's mean radius is approximately 6,371 kilometers."}`

🔧 **Action:** `convert_units({"value": 1, "from_unit": "kilometers", "to_unit": "miles"})`
📋 **Observation:** `{"error": "Unsupported conversion: kilometers \u2192 miles"}`


---

**Step 2 — Thought:**
I successfully retrieved Earth's radius in kilometers, which is approximately 6,371 km. However, I encountered an issue with converting kilometers to miles directly. 

I'll now convert 6371 km to miles using the conversion factor: 1 km ≈ 0.621371 miles. I'll perform this calculation next.

🔧 **Action:** `calculate({"expression":"6371 * 0.621371"})`
📋 **Observation:** `{"expression": "6371 * 0.621371", "result": 3958.754641}`


---

**⚠️ Max iterations (2) reached.**

---

With only 2 iterations, the agent couldn't complete all the steps. Instead of looping forever, it stopped and reported what it had so far. This is the stop condition doing its job — a critical safety mechanism for any agent.

## Part 9: Test — Short-Term Memory (Multi-Turn)

Since our agent maintains a `messages` list, it has short-term memory across turns. Let's verify it remembers earlier results.

In [23]:
memory_agent = ResearchAgent()

md("### Turn 1")
answer = memory_agent.chat("What is the boiling point of water in Fahrenheit?")
md(f"**Agent:** {answer}\n")

md("### Turn 2 (references Turn 1)")
answer = memory_agent.chat("Convert that temperature to Celsius.")
md(f"**Agent:** {answer}\n")

md("### Turn 3 (references Turn 2)")
answer = memory_agent.chat("Now multiply that Celsius value by 3.")
md(f"**Agent:** {answer}")

### Turn 1

**Agent:** The boiling point of water is 100°C, which is equivalent to 212°F.


### Turn 2 (references Turn 1)

**Agent:** The temperature of 212°F converts to 100°C.


### Turn 3 (references Turn 2)

**Agent:** Multiplying 100°C by 3 gives 300°C.

The agent correctly resolved "that temperature" and "that Celsius value" by referencing its conversation history — short-term memory at work.

## Part 10: Full Integration Test

One final complex question that exercises everything: planning, multiple tools, chained results, and reflection.

In [24]:
agent_final = ResearchAgent()

question = (
    "I want to know: if I could drive to the Moon at 100 km/h, "
    "how many days would it take? "
    "Also tell me that same distance in miles."
)

md(f"### 🧑 User\n{question}\n")
answer = agent_final.chat(question)

md("### 🔍 Full Reasoning Trace\n")
agent_final.show_trace()

### 🧑 User
I want to know: if I could drive to the Moon at 100 km/h, how many days would it take? Also tell me that same distance in miles.


### 🔍 Full Reasoning Trace


**Step 1 — Thought:**
PLAN:
1. Look up the average distance from Earth to the Moon.
2. Calculate the time it would take to travel that distance at 100 km/h.
3. Convert the distance from kilometers to miles.

EXECUTE:
- Lookup the distance from Earth to the Moon.
- Use the distance and speed to find travel time.
- Convert distance to miles.

Let's start by finding the distance from Earth to the Moon. (Usually about 384,400 km on average)

I'll confirm this fact now.

🔧 **Action:** `lookup_fact({"topic":"moon distance"})`
📋 **Observation:** `{"topic": "moon distance", "fact": "The average distance from Earth to the Moon is about 384,400 kilometers."}`


---

**Step 2:**

🔧 **Action:** `convert_units({"value":384400,"from_unit":"kilometers","to_unit":"miles"})`
📋 **Observation:** `{"error": "Unsupported conversion: kilometers \u2192 miles"}`


...

---

**Step 6 — Thought:**
It would take approximately 3844 hours to drive to the Moon at 100 km/h.

Finally, to convert this into days, divide by 24 hours/day.

🔧 **Action:** `calculate({"expression":"3844 / 24"})`
📋 **Observation:** `{"expression": "3844 / 24", "result": 160.166667}`


---

**Step 7 — Final Answer:**
It would take approximately 160 days to drive from Earth to the Moon at 100 km/h.

### Summary:
- Distance from Earth to Moon: about 238,856 miles
- Time to drive at 100 km/h (~62.14 mph): approximately 160 days

---

## 🎯 Summary

### What You Built

You built a complete tool-using research agent that plans before acting, reasons transparently at each step (ReAct), uses multiple tools, recovers from errors, reflects on its answers, respects guardrails, and enforces stop conditions — all with plain OpenAI SDK and no external frameworks.

### Key Takeaways

1. **Planning & task decomposition** — asking the LLM to break a complex question into numbered sub-tasks before acting leads to more organized and reliable multi-step solutions
2. **ReAct reasoning** — the Thought → Action → Observation cycle makes the agent's decision-making transparent and debuggable, compared to a plain agent loop where reasoning is invisible
3. **Error recovery** — wrapping tool execution in try/except and returning structured error JSON lets the LLM adapt instead of crashing; the agent can explain the failure or try an alternative
4. **Reflection** — prompting the agent to review its work before answering catches mistakes from rounding, wrong units, or missed sub-tasks
5. **Stop conditions** — enforcing a maximum iteration limit prevents infinite loops; the agent reports partial progress instead of running forever
6. **Guardrails** — clear instructions in the system prompt about what the agent must refuse are a simple but effective first line of defense for agent safety